# ELPC Manure Spotter

This notebook runs code on Google Earth Engine to detect manure spread on
snow-covered croplands in the upper midwest.

**What it produces**

* A GeoJSON file of detection polygons in your Google Drive, with columns for
  county, acres, latitude and longitude, sensor, and whether the detection
  touches a spread already found earlier in the season.
* Optionally, CSV file of all detections, with the same information the GeoJSON
* Optionally, a labeled satellite image of each detection, so you can inspect
  the results visually.

A detection is not confirmation that manure was applied. Each detection
requires visual review of the imagery to rule out false positives, and ideally
on-the-ground inspection.

A full description of how the workflow works can be found in the open-access article
["Near-real-time detection of manure applications on snow-covered croplands in Google Earth Engine using Sentinel-2 and Landsat-8/9"](https://doi.org/10.1016/j.rsase.2026.102141),
published 2026 in *Remote Sensing Applications: Society and Environment*.

---

## What you need before you start

**Before running this workflow, you must set up these four things** (if any one is missing, the notebook stops and tells you which one):

**1. A Google account.** The notebook reads and writes files in your Google
Drive.

**2. A registered Earth Engine account** tied to that same Google account. Sign
up at https://earthengine.google.com/signup/. Registration gives you a project ID, which you
will paste into section 0. You can find it in the top-right corner of
https://code.earthengine.google.com/.

**3. Google Colab**, at https://colab.research.google.com/. Signing in with
your Google account is the whole setup. There is nothing to install: Colab is a
website, and the packages this notebook needs are installed by the first code
cell every time you run it.

**4. The training data**, downloaded from Zenodo
(https://doi.org/10.5281/zenodo.18225627) and copied into your Google Drive.
The download is six files:

    s2_groundtruth.gpkg    s2_non_manure.gpkg    s2_sat_manure.gpkg
    l89_groundtruth.gpkg   l89_non_manure.gpkg   l89_sat_manure.gpkg

Put all six in one folder in My Drive, leaving the file names exactly as they
came, and set `DRIVE_DATA_FOLDER` in section 0 to that folder. The default is
`manure_spotter/training_data`, meaning `My Drive/manure_spotter/training_data`.
There are no subfolders.

---

## Running it
The only cell you need to touch is the top cell titled "0. User Configuration" Once you've adjusted it to your needs, choose **Runtime → Run all**.

At minimum, set:

- `GEE_PROJECT_NAME` — your Earth Engine project ID.
- `TARGET_DATE` — the date to analyze, written year-month-day: `'2026-02-15'`.
- `FOCAL_STATE` — the state to work in, as a two-letter postal abbreviation.
  One run covers one state. At this time, the workflow does not support multi-state analysis
- `FOCAL_COUNTIES` — the counties to analyze, by name. The default is the
  Wisconsin study area from the paper. Fewer counties run faster, so
  a single county is the quickest way to confirm your setup works before
  committing to a full run. County names have to match the Census spelling, but case does not matter. If a county name is not found, the workflow will suggest the county with the most simmilar name

Two permission windows appear during the run, one for Earth Engine and one for
Google Drive. The notebook waits without printing anything until you click
through both, so if it looks stuck, check for a popup or an authorization link
in the cell output.

### Initial set up takes two runs

The notebook needs to build a snow-free autumn composite image before it can search for manure, so the first run will focus on building that, but subsequent runs will find manure.

**Run 1 — build the autumn composites.** The notebook works out which of your
counties still need composite imagery, starts a build for each one, and stops.
There is nothing to paste anywhere. Watch the builds finish in the Tasks tab of
the Earth Engine Code Editor (https://code.earthengine.google.com/), then run
the notebook again.

**Run 2 — train the classifiers and get your first detections.** The notebook
trains a Random Forest classifier for each satellite from your training data,
saves each one to your project, prints its name, and carries straight on to
detection. Paste the two classifier names into `S2_CLASSIFIER_ASSET` and
`L89_CLASSIFIER_ASSET`.

**Run 3 and after.** Everything is loaded rather than built, and changing
`TARGET_DATE` is all a normal run takes.

Composites are stored one image per county per autumn, in a collection in your
own Earth Engine project. Each run mosaics together the counties it needs. Add a
county later and only that county gets built — the rest are reused, and nothing
has to be renamed or repasted.

Splitting them by county is safe because every step of the composite is computed
one pixel at a time, so a mosaic of per-county images is identical to a single
image built across all of them at once.

The composites are built from October 15 through January 31. That window is
fixed in the settings below section 0 rather than exposed as an option, because
it has to match what the classifier was trained against. The years are filled in
for you: the target composite follows `TARGET_DATE`, so changing the winter you
analyze is enough.

### Counties outside the study area

The notebook reads the training polygons to work out which counties they cover,
and that is the region the classifier has been validated in — with the Zenodo
data, eighteen Wisconsin counties. Nothing is written down, so if you replace
the training data with polygons from somewhere else, the tested region and the
training composite both follow it.

You can point the notebook at any state by changing `FOCAL_STATE` and
`FOCAL_COUNTIES`, and it will run, but it has not been validated outside that
region. The notebook
prints a warning when your counties fall outside it.

---

## Reading the output

While it runs, the notebook reports how much of your area each satellite
covered, how much of that was clear of clouds, and how much was snow-covered.
Those three numbers explain why a date returns few detections, or none.

If you see

    no images had suitable atmospheric and surface conditions for inferencing

the date had too many clouds, too little snow snow-free, or else it had no satellite coverage.
Nothing went wrong. Try another date.

When detections are found, the notebook hands the export job to Earth Engine
and finishes. **The file does not appear immediately.** Watch the Tasks tab of
the Earth Engine Code Editor (https://code.earthengine.google.com/) for
progress. When the task completes, the GeoJSON appears in

    My Drive/manure_spotter_table_exports/

Image chips, if you turned them on, go to `My Drive/GEE_snapshot_exports/`.

Open the GeoJSON in QGIS, or drag it onto https://geojson.io, to see the
detections on a map.


# 0. User Configuration (required)

In [ ]:
# =============================================================================
#  0. USER CONFIGURATION
#  This is the only cell you need to edit. Everything below it can be left
#  alone. Run the notebook with Runtime -> Run all.
# =============================================================================

# --- REQUIRED ----------------------------------------------------------------

# Your Earth Engine project ID. After your Earth Engine registration is
# approved, you can find it in the top-right corner of
# https://code.earthengine.google.com/     Example: "manure-spotter-2026"
GEE_PROJECT_NAME = "example-gee-project"

# The date you want to analyze, written year-month-day.
TARGET_DATE = "2026-02-02"

# The state to work in, as a two-letter postal abbreviation. One run covers one
# state. To analyze a different state, change this and run again.
FOCAL_STATE = "WI"

# The counties to analyze, by name. The default is the Wisconsin study area
# from the paper. Fewer counties run faster, so a single county is a good way
# to check that your setup works before running the whole area.
#
# Names have to match the Census spelling, but you do not have to get it right
# the first time: if one does not match, the notebook stops and tells you which
# one, with the closest spellings it found.
FOCAL_COUNTIES = ["St Croix"]
#    "Barron", "Brown", "Buffalo", "Burnett", "Calumet", "Columbia",
#    "Dane", "Door", "Dunn", "Green", "Kewaunee", "Manitowoc",
#    "Outagamie", "Pepin", "Pierce", "Polk"]

# The folder in your Google Drive holding the training data downloaded from
# Zenodo (https://doi.org/10.5281/zenodo.18225627). The path is relative to
# "My Drive". Put all six .gpkg files directly in it, with the names they came
# with, and no subfolders.
DRIVE_DATA_FOLDER = "manure_spotter/training_data"


# --- SAVED CLASSIFIERS (leave blank on your first run) -----------------------
# Training takes a while. Leave these blank and the notebook trains a
# classifier for each satellite, saves it to your project, and prints the name.
# Paste those names in here to skip training on later runs. Either a bare name
# ("rf_s2_846_polys_20260215") or a full path works.
#
# The autumn composites need no setting at all — the notebook keeps them in a
# collection in your project and finds them by county and year.

S2_CLASSIFIER_ASSET  = "example-s2-classifer"
L89_CLASSIFIER_ASSET = "example-l89-classifier"


# --- OPTIONS -----------------------------------------------------------------

# How much detail to print while running. "PROGRESS" is the normal setting.
# Switch to "INFO" or "DEBUG" when something looks wrong and you want more.
LOG_LEVEL = "PROGRESS"

# Also export a labeled satellite image of every detection to your Drive.
# This adds substantial time to the run.
EXPORT_IMAGE_CHIPS = True
EXPORT_EXCEL = True


# 1.&nbsp;Install and Import Statements

In [ ]:
!pip -q install geemap geopandas shapely

In [ ]:
# PEP 8 imports: std-lib first, then 3rd-party alphabetized within each group.

from datetime import datetime, timedelta
import html
import json
import pandas as pd
import pprint
import sys
import logging
from dataclasses import dataclass, field
from collections.abc import Callable
from difflib import get_close_matches
from pathlib import Path
import time
import re

import ee
import geemap
import geopandas as gpd
from google.colab import drive, output


def _announce_stop(message: str) -> None:
    """
    Put the reason a run stopped somewhere it cannot be scrolled past.

    A Run all can end anywhere in a long notebook, and the message explaining
    why is easy to miss — particularly after a composite build that ran for
    hours while nobody was watching. So it goes in two places: a banner in the
    cell output, and a browser dialog that shows up wherever you happen to be
    on the page.
    """
    from IPython.display import display, HTML

    display(HTML(
        "<div style='border-left:6px solid #b3261e;background:#fceeed;"
        "padding:16px 20px;margin:10px 0;border-radius:4px;"
        "font-family:system-ui,-apple-system,sans-serif;font-size:14px;"
        "line-height:1.5;color:#1b1b1b;white-space:pre-wrap'>"
        "<div style='font-weight:600;font-size:15px;color:#b3261e;"
        "margin-bottom:8px'>Manure Spotter stopped</div>"
        f"{html.escape(message)}"
        "<div style='margin-top:12px;font-size:12px;color:#5f6368'>"
        "The red text below this box is only Python's record of where it "
        "stopped. The explanation is here.</div></div>"
    ))

    paragraphs = message.split("\n\n")
    dialog_text = "\n\n".join(paragraphs[:2])
    if len(dialog_text) > 400:
        dialog_text = dialog_text[:400].rstrip() + "…"
    if dialog_text != message:
        dialog_text += "\n\n(full details in the notebook)"

    output.eval_js(
        "alert(" + json.dumps("Manure Spotter stopped\n\n" + dialog_text) + ");",
        ignore_result=True,
    )


class ManureSpotterAbort(RuntimeError):
    """
    Stops the run and explains why in plain language. If you see one of these,
    read the message: it will tell you which setting in section 0 to change.

    Creating one also puts the message on screen as a banner and, in Colab, a
    browser dialog, so a run that stops while you are away from the screen
    still tells you why when you come back.
    """

    def __init__(self, message: str):
        super().__init__(message)
        try:
            _announce_stop(message)
        except Exception:
            pass          # a missing banner must never hide the message itself

In [ ]:
GEE_PROJECT_NAME = GEE_PROJECT_NAME.strip()

if not GEE_PROJECT_NAME:
    raise ManureSpotterAbort(
        "GEE_PROJECT_NAME is blank.\n"
        "Go back to section 0 and enter your Earth Engine project ID. You can "
        "find it in the top-right corner of https://code.earthengine.google.com/ "
        "once your Earth Engine registration has been approved."
    )

try:
    datetime.strptime(TARGET_DATE, "%Y-%m-%d")
except ValueError:
    raise ManureSpotterAbort(
        f"TARGET_DATE is '{TARGET_DATE}', which is not a date this notebook "
        "can read. Write it as year-month-day, for example '2026-02-15'."
    ) from None

# A permission window will open. Click through it to let the notebook talk to
# Earth Engine on your behalf.
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_NAME)


# 2.&nbsp;Configuration

## 2.1 Internal settings

You do not need to edit anything from here on. Choose Runtime -> Run all.

In [ ]:
# ---- this can be swapped out for any FC
# ---------------------------------------------------------------------------
# Turn the county names from section 0 into Census GEOIDs.
#
# Filtering on names directly would fail quietly: a misspelling would just
# match nothing and you would analyze a smaller area without being told. So
# every name is checked against the Census county list for FOCAL_STATE, and a
# name that does not match stops the run instead.
#
# One run covers one state. Counties are looked up inside FOCAL_STATE only, so
# a county from a neighbouring state is reported as not found rather than
# silently pulled in.
# ---------------------------------------------------------------------------

_COUNTIES_FC = ee.FeatureCollection("TIGER/2018/Counties")
_STATES_FC   = ee.FeatureCollection("TIGER/2018/States")


def _resolve_counties(state: str,
                      names: list[str]) -> tuple[list[str], dict[str, str]]:
    """
    Take a state postal abbreviation and a list of county names, and return the
    matching GEOIDs plus a lookup from GEOID back to a readable label.

    County names are not unique inside every state. Maryland, Missouri and
    Virginia each have an independent city sharing its name with a county —
    'Fairfax city' and 'Fairfax County' are different places. Matching on the
    short name alone would quietly pick one of them, so an ambiguous name stops
    the run and asks for the longer form.
    """

    def _normalize_county_name(name: str) -> str:
        """Casefold and collapse punctuation/spaces, so 'St. Croix' and
        'St Croix' (or extra whitespace) match the same county."""
        return re.sub(r"[^a-z0-9]+", " ", name.casefold()).strip()

    state = str(state).strip().upper()
    names = [str(n).strip() for n in names if str(n).strip()]

    if not state:
        raise ManureSpotterAbort(
            "FOCAL_STATE is blank. Enter a two-letter state postal "
            "abbreviation in section 0, for example 'WI'."
        )

    if not names:
        raise ManureSpotterAbort(
            "FOCAL_COUNTIES is empty. Add at least one county name in "
            'section 0, for example:\n\n    FOCAL_COUNTIES = ["Kewaunee"]'
        )

    # Postal abbreviation to state FIPS, read from the Census state table
    # rather than hardcoded, so it cannot drift.
    state_fips = dict(zip(
        _STATES_FC.aggregate_array("STUSPS").getInfo(),
        _STATES_FC.aggregate_array("STATEFP").getInfo(),
    ))

    if state not in state_fips:
        raise ManureSpotterAbort(
            f"FOCAL_STATE is '{state}', which is not a state postal "
            "abbreviation. Use two letters, like 'WI'."
        )

    in_state = _COUNTIES_FC.filter(ee.Filter.eq("STATEFP", state_fips[state]))
    census_short = in_state.aggregate_array("NAME").getInfo()
    census_full  = in_state.aggregate_array("NAMELSAD").getInfo()
    census_geoid = in_state.aggregate_array("GEOID").getInfo()

    by_full: dict[str, str]        = {}   # 'fairfax county' -> GEOID
    by_short: dict[str, list[str]] = {}   # 'fairfax'        -> [GEOID, GEOID]
    pretty: dict[str, tuple[str, str]] = {}

    for short, full, geoid in zip(census_short, census_full, census_geoid):
        by_full[_normalize_county_name(full)] = geoid
        by_short.setdefault(_normalize_county_name(short), []).append(geoid)
        pretty[geoid] = (short, full)

    # Label every county in the state, not just the chosen ones, so messages
    # about the training counties read as names too.
    labels_all = {g: f"{p[0]}, {state}" for g, p in pretty.items()}

    geoids: list[str] = []
    labels: dict[str, str] = dict(labels_all)
    problems: list[str] = []

    for name in names:
        key = _normalize_county_name(name)

        if key in by_full:
            geoid = by_full[key]

        elif key in by_short and len(by_short[key]) == 1:
            geoid = by_short[key][0]

        elif key in by_short:
            options = " or ".join(f"'{pretty[g][1]}'" for g in by_short[key])
            problems.append(
                f"  '{name}' is ambiguous in {state}. Write it as {options}."
            )
            continue

        else:

            matches = get_close_matches(
                _normalize_county_name(name), list(by_full) + list(by_short), n=6, cutoff=0.6)

            suggested: list[str] = []
            for candidate in matches:
                owners = ([by_full[candidate]] if candidate in by_full
                          else by_short.get(candidate, []))
                for geoid in owners:
                    if geoid not in suggested:
                        suggested.append(geoid)
                if len(suggested) >= 3:
                    break
            suggested = suggested[:3]

            # Offer the short name, except for the rare shared ones where only
            # the long form picks out a single place.
            spellings = [
                pretty[g][0] if len(by_short[_normalize_county_name(pretty[g][0])]) == 1
                else pretty[g][1]
                for g in suggested
            ]

            hint = (" Did you mean " + " or ".join(f"'{s}'" for s in spellings)
                    + "?" if spellings else "")

            problems.append(
                f"  there is no county named '{name}' in {state}.{hint}")
            continue

        geoids.append(geoid)
        labels[geoid] = f"{pretty[geoid][0]}, {state}"

    if problems:
        raise ManureSpotterAbort(
            f"Some of the county names in section 0 could not be matched in "
            f"{state}:\n\n"
            + "\n".join(problems)
            + "\n\nNames have to match the Census spelling, including "
              "punctuation and spaces. Capitalization does not matter. If one "
              "of these counties is in a different state, note that a run "
              "covers one state only."
        )

    return sorted(set(geoids)), labels


FOCAL_COUNTY_IDS, _COUNTY_LABELS = _resolve_counties(FOCAL_STATE, FOCAL_COUNTIES)

print(f"area of interest: {len(FOCAL_COUNTY_IDS)} counties in "
      f"{str(FOCAL_STATE).strip().upper()} — "
      + ", ".join(_COUNTY_LABELS[g].rsplit(",", 1)[0] for g in FOCAL_COUNTY_IDS))

AOI_FC: ee.FeatureCollection = (_COUNTIES_FC
      .filter(ee.Filter.inList("GEOID", FOCAL_COUNTY_IDS))
)

FOCAL_STATE = str(FOCAL_STATE).strip().upper()

# ---------------------------------------------------------------------------
# Translate the plain settings from section 0 into the paths and date ranges
# the rest of the notebook expects.
# ---------------------------------------------------------------------------

GEE_ASSET_ROOT = Path(f"projects/{GEE_PROJECT_NAME}/assets")


def _qualify_asset(name: str | None) -> Path | None:
    """
    Accept either a bare asset name ('rf_s2_846_polys_20260215') or a full
    Earth Engine path ('projects/a-project/assets/rf_s2_846_polys_20260215')
    and return a full path. A blank setting returns None, which tells the
    notebook to build that item instead of loading it.
    """
    if not name or not str(name).strip():
        return None
    name = str(name).strip()
    return Path(name) if name.startswith("projects/") else GEE_ASSET_ROOT / name


# The stretch of autumn and early winter the composites are built from, as
# month-day pairs. This is a property of the method, not a user preference:
# it has to match what the classifier was trained against, so changing it
# invalidates a trained classifier. The years are filled in automatically.
# A window ending earlier in the calendar than it starts runs into the
# following year, which is what this default does.
COMPOSITE_WINDOW: tuple[str, str] = ("10-15", "01-31")


def _fall_window(fall_year: int,
                 window: tuple[str, str] = COMPOSITE_WINDOW) -> tuple[str, str]:
    """
    Put real years on COMPOSITE_WINDOW for a given autumn. A window whose end
    falls earlier in the calendar than its start runs into the next year, so
    ("10-15", "01-31") over autumn 2025 gives 2025-10-15 to 2026-01-31.
    """
    start_md, end_md = window
    end_year = fall_year if end_md > start_md else fall_year + 1
    start, end = f"{fall_year}-{start_md}", f"{end_year}-{end_md}"

    for value in (start, end):
        try:
            datetime.strptime(value, "%Y-%m-%d")
        except ValueError:
            raise ManureSpotterAbort(
                f"COMPOSITE_WINDOW produced '{value}', which is not a real "
                'date. Write each half as month-day, for example ("10-15", '
                '"01-31").'
            ) from None

    return start, end


def _fall_year_for(target_date: str) -> int:
    """
    The autumn preceding the winter a date falls in. February 2026 belongs to
    the winter that began in autumn 2025, so this returns 2025. Deriving it
    means you do not have to remember to change the composite when you change
    the year.
    """
    d = datetime.strptime(target_date, "%Y-%m-%d")
    return d.year - 1 if d.month <= 6 else d.year


@dataclass(frozen=True)
class TrainingDataSummary:
    """What the training files turn out to contain, read from the files."""
    county_ids   : list[str]
    fall_years   : list[int]
    polygon_count: int
    first_date   : str
    last_date    : str

@dataclass(frozen=True)
class DataImportConfig:
    id_col: str   = "id"
    date_col: str = "use_date"

    type_col: str = "type"
    manure_label: str      = "manure"
    non_manure_label: str  = "fp"
    class_col: str = "class"

    # Parsing/normalization for Python-side dates in training files
    py_date_in: str  = "%Y-%m-%d"     # what you expect to read
    py_date_out: str = "%Y-%m-%d"     # how you normalize/store

    # Behavior toggles
    enforce_unique_ids: bool = True
    geodesic: bool           = True
    copy_gdf: bool           = True


@dataclass(frozen=True) # for reporting duplicate IDs in error
class DuplicateIDsReport:
    ids: list[str]
    rows: gpd.GeoDataFrame
    def is_empty(self) -> bool: return not self.ids
    def head(self, n: int = 20): return self.rows.head(n)

class DuplicateIDsError(ValueError):
    def __init__(self, message: str, report: DuplicateIDsReport):
        super().__init__(message)
        self.report = report

@dataclass(frozen=True)
class DateFormats:
    # Python strftime/strptime
    py_iso: str     = "%Y-%m-%d"
    py_compact: str = "%Y%m%d"
    # Earth Engine (Joda-time) patterns
    ee_iso: str     = "yyyy-MM-dd"
    ee_compact: str = "yyyyMMdd"

@dataclass(frozen=True)
class Paths:
    gt: Path
    so: Path
    sm: Path
    rf: Path | None

@dataclass(frozen=True)
class Kernel:
    size: int
    shape: str
    units: str

@dataclass(frozen=True)
class SensorConfig:
    pretty          : str
    ic              : ee.ImageCollection
    optical_bands   : list[str]
    band_lookup     : dict[str, str]
    cloud_band      : str
    vis_brightness_thresh     : float
    clear_cluster   : int
    cloud_kernel    : Kernel
    paths           : Paths
    scalar          : float
    cloud_pct_prop  : str | None
    cirrus_pct_prop : str | None
    min_area_after_subtraction: int ## if there is a prior dection near the new detection, how large must the area of the new detection be?
    rgb_scale       : int
    export_scale    : int
    cluster_thresh  : int


@dataclass(frozen=True)
class RFParams:
    num_trees: int = 400
    vars_per_split: int = 5
    min_leaf_pop: int = 5
    bag_frac: float = 0.6
    max_nodes: int | None = None
    seed: int = 0


@dataclass(frozen=True)
class FeatureExportParams:
# ---- feature collection property names ----
# ---- core properties ----
    sensor_prop_name: str = "sensor"
    date_prop_name: str = "detection_date"
    id_prop_name: str = "id"

    prior_spread_adjacency_prop_name: str = "touches_prior_spread"

# ---- core property format ---
    sensor_prop_delimiter: str = "_"
    date_fmt: str = "MMddYYYY"
    pad: int = 4 #leading zeros for ID

# ---- enriched properties ----
    county_prop_name: str = "county"
    lat_prop_name: str = "lat"
    lon_prop_name: str = "lon"
    acres_prop_name: str = "acres"

    ## compute list of tuble of export properties
    @property
    def export_fields(self) -> tuple[str, ...]:
        return (
            self.id_prop_name,
            self.sensor_prop_name,
            self.date_prop_name,
            self.county_prop_name,
            self.lat_prop_name,
            self.lon_prop_name,
            self.acres_prop_name,
            self.prior_spread_adjacency_prop_name
        )

#------image export properties
    bands: tuple[str,str,str] = 'red', 'green', 'blue'
    pct_low: int = 2
    pct_high: int = 98
    gamma: float = 1.2
    interpolation: str = 'bicubic'
    buffer: int = 30
    export_folder: str = "GEE_snapshot_exports"
    crs: str = 'EPSG:3857'
    dimension: int = 1200
    bbox_thickness: int = 4
    bbox_color: str = 'FF00FF'

@dataclass
class FallCompositeResult:
    img: ee.Image | None
    messages: list[str]
    tasks: list = field(default_factory=list)


@dataclass(frozen=True)
class GlobalConfig:

    ############################
    ## USER-SET CONFIGURATION ##
    ############################

    target_date    : str | None = None

    log_level      : str = "INFO"

    export_image_chips: bool = EXPORT_IMAGE_CHIPS

    # Everything below is assembled from section 0 — nothing points at anyone
    # else's Earth Engine project.
    gee_asset_root : Path = GEE_ASSET_ROOT

    # Autumn composites live one-image-per-county inside this collection, and
    # are found by county and year rather than named in section 0.
    fall_composite_collection : str = "manure_spotter_fall_composites"

    mount_point: str = "/content/drive"
    drive_root : str = "MyDrive"
    drive_base : str = DRIVE_DATA_FOLDER
    table_export_folder: str = "manure_spotter_table_exports"

    # File names as they come out of the Zenodo download, sitting directly in
    # DRIVE_DATA_FOLDER with no subfolders.
    s2_paths       : Paths = field(default_factory=lambda: Paths(
        gt=Path("s2_groundtruth.gpkg"),
        so=Path("s2_non_manure.gpkg"),
        sm=Path("s2_sat_manure.gpkg"),
        rf=_qualify_asset(S2_CLASSIFIER_ASSET),
    ))
    l89_paths      : Paths = field(default_factory=lambda: Paths(
        gt=Path("l89_groundtruth.gpkg"),
        so=Path("l89_non_manure.gpkg"),
        sm=Path("l89_sat_manure.gpkg"),
        rf=_qualify_asset(L89_CLASSIFIER_ASSET),
    ))

    ############################
    #### DEFAULT PARAMETERS ####
    ############################

    post_detection_suppression_days : int = 9 #9 is prob best after a detection is found in a location, algorithm will ignore that location for this number of days.

    # Built from COMPOSITE_WINDOW when CFG is created, at the bottom of this
    # cell. The training window is not here: it comes from the dates inside the
    # training files, which are not readable until Drive has been mounted.
    target_fall_comp_rng    : tuple[str, str] | None = None

    composite_window : tuple[str, str] = COMPOSITE_WINDOW

    date_format    : DateFormats = field(default_factory=DateFormats)

    data_import_params: DataImportConfig = DataImportConfig()

    max_cloud_cover: int = 90

    min_masked_pct : int = 1 # 1-100. loop will terminate if snow_crop_cloud mask is below this

    rf_params      : RFParams = RFParams()

    coarse_scale   : int = 200
    road_buffer    : int = 30

    noise_thresh   : int = 10

    post_kernel    : Kernel = Kernel(10, "square", "meters") ## controls how much larger the vector is than the actual classified pixels

    geo_blocks     : str = "TIGER/2020/BG" #"TIGER/2010/Blocks" ##

    feature_export_params: FeatureExportParams = FeatureExportParams()
    export_prefix  : str = "manure_detections_v22"
    poll_seconds   : int = 120
    timeout_minutes: int = 40


    ############################
    # CONSTANTS (DO NOT TOUCH) #
    ############################

    nbsi_constant  : float = 0.36

    m2acres        : ee.Number = field(
        default_factory=lambda: ee.Number(0.000_247_105_381)
        )

    counties_all: ee.FeatureCollection = field(
        default_factory=lambda: ee.FeatureCollection("TIGER/2018/Counties")
    )

    ############################
    ### METHODS & PROPERTIES ###
    ############################

    @property
    def drive_root_path(self) -> Path:
        return Path(self.mount_point) / Path(self.drive_root)

    @property
    def fall_composite_path(self) -> Path:
        return self.gee_asset_root / self.fall_composite_collection

    @property
    def data_root(self) -> Path:
        return self.drive_root_path / Path(self.drive_base)

    @property
    def table_export_path(self) -> Path:
        return self.drive_root_path / self.table_export_folder

    def ensure_table_export_path(self) -> Path:
        p = self.table_export_path
        p.mkdir(parents=True, exist_ok=True)
        return p

    def resolve(self, p: str | Path) -> Path:
        p = Path(p)
        return p if p.is_absolute() else (self.data_root / p)


CFG = GlobalConfig(
    target_date          = TARGET_DATE,
    log_level            = LOG_LEVEL,
    target_fall_comp_rng = _fall_window(_fall_year_for(TARGET_DATE)),
)

SENSORS = {
    "s2": SensorConfig(
        pretty="Sentinel‑2",
        ic = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED"),
        optical_bands=[
            "B2","B3","B4","B5","B6","B7",
            "B8","B8A","B9","B11","B12","B1"
        ],
        band_lookup={
            "aerosols":"B1", "blue":"B2", "green":"B3", "red":"B4",
            "red edge 1":"B5", "red edge 2":"B6", "red edge 3":"B7",
            "nir":"B8", "red edge 4":"B8A", "water vapour":"B9",
            "swir 1":"B11", "swir 2":"B12"
        },
        cloud_band="SCL",
        vis_brightness_thresh= .644, #.608, # 0.701476,
        clear_cluster =400,
        cloud_kernel= Kernel(3, "square", "pixels"),
        paths=CFG.s2_paths,
        scalar=0.0001,
        cloud_pct_prop= 'CLOUDY_PIXEL_PERCENTAGE',
        cirrus_pct_prop= 'THIN_CIRRUS_PERCENTAGE',
        min_area_after_subtraction = 3000, # ~50% of main 60 pixel thresh
        rgb_scale = 10,
        export_scale = 3,
        cluster_thresh = 65
    ),

    "l89": SensorConfig(
        pretty="Landsat‑8/9",
        ic = (ee.ImageCollection("LANDSAT/LC08/C02/T1_RT_TOA")
            .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA"))
            .merge(ee.ImageCollection('LANDSAT/LC09/C02/T2_TOA'))
        ),
        optical_bands=[
            "B2","B3","B4","B5","B6","B7",
            "B8","B9","B10","B11","B1"
        ],
        band_lookup={
            "aerosols":"B1", "blue":"B2", "green":"B3", "red":"B4",
            "nir":"B5", "swir 1":"B6", "swir 2":"B7", "panchromatic":"B8",
            "cirrus":"B9", "thermal 1":"B10", "thermal 2":"B11"
        },
        cloud_band="QA_PIXEL",
        vis_brightness_thresh= .55, #0.701476, #.55 (experimental)
        clear_cluster =400,
        cloud_kernel=Kernel(0, "square", "pixels"),
        paths=CFG.l89_paths,
        scalar=1,
        cloud_pct_prop = 'CLOUD_COVER_LAND',
        cirrus_pct_prop = None,
        min_area_after_subtraction = 6000, # ~100% of main 60 pixel thresh
        rgb_scale = 30,
        export_scale = 3,
        cluster_thresh = 85

    ),
}



@dataclass(frozen=True)
class CloudCullResult:
    filtered: dict[str, ee.ImageCollection]
    available: list[str]
    counts: dict[str, int]

@dataclass(frozen=True)
class AOIContext:
    counties: ee.FeatureCollection
    date_str: str
    ee_date: ee.Date
    geom: ee.Geometry
    acres: ee.Number
    m2acres: ee.Number

# if counties are used to define AOI, aoi_fc and counties are the same variable
def make_aoi_context( aoi_fc: ee.FeatureCollection,
                      *,
                      date_str: str,
                      m2acres: ee.Number,
                      counties: ee.FeatureCollection) -> AOIContext:
    geom  = aoi_fc.geometry().dissolve()
    acres = geom.area(maxError=1000).multiply(m2acres)
    return AOIContext(
        date_str=date_str,
        ee_date=ee.Date(date_str),
        geom=geom,
        acres=acres,
        m2acres=m2acres,
        counties=counties
    )

@dataclass(frozen=True)
class SensorCoverage:
    date: str
    sensor: str
    geom: ee.Geometry
    mask: ee.Image
    acres: ee.Number
    percent_aoi: ee.Number
    m2acres: ee.Number = CFG.m2acres

@dataclass(frozen=True)
class ClearskyMaskParams:
    sensor_pretty   : str
    coarse_scale    : int
    buffer_kernel   : Kernel
    cluster_thresh  : int

@dataclass(frozen=True)
class ClearskyMaskResult:
    mask: ee.Image
    acres: ee.Number
    coverage: SensorCoverage
    params: ClearskyMaskParams

    @property
    def clear_pct_of_coverage(self) -> ee.Number:
        return self.acres.divide(self.coverage.acres).multiply(100)

@dataclass(frozen=True)
class VisBrightnessParams:
    band_mapping: dict[str, str]
    vis_brightness_thresh: float
    coarse_scale: int
    geo_blocks: ee.FeatureCollection
    k: float = CFG.nbsi_constant

@dataclass(frozen=True)
class VisBrightnessOutput:
    img: ee.Image
    stats: ee.FeatureCollection
    high_fc: ee.FeatureCollection
    high_mask: ee.Image
    pct_high: ee.Number
    high_acres: ee.Number

@dataclass
class SensorOutputs:

    ic              : ee.ImageCollection
    aoi_coverage    : SensorCoverage
    clearsky        : ClearskyMaskResult
    crop_mask       : ee.Image
    vis_brightness  : VisBrightnessOutput

    inference_bands : list[str] | None
    target_image    : ee.Image | None
    rf_output       : ee.Image | None
    noise_removed   : ee.Image | None
    detection_polys : ee.FeatureCollection | None


## 2.2 Logging Setup

In [ ]:
PROGRESS_LEVEL = 15
logging.addLevelName(PROGRESS_LEVEL, "PROGRESS")

def _resolve_level(name: str) -> int:
    """Map a level name/string to its numeric value, incl. custom 'PROGRESS'."""
    if name.upper() == "PROGRESS":
        return PROGRESS_LEVEL
    return getattr(logging, name.upper(), logging.INFO)

logging.basicConfig(
    level= _resolve_level(CFG.log_level),
    format="%(asctime)s — %(levelname)s — %(message)s",
    datefmt = "%Y-%m-%d %H:%M",
    stream=sys.stdout,
    force=True
)
logger = logging.getLogger(__name__)


def progress(msg: str, *args, **kwargs) -> None:
    if logger.isEnabledFor(PROGRESS_LEVEL):
        logger.log(PROGRESS_LEVEL, msg, *args, **kwargs)


def ee2float(obj, *, prec: int = 2, level: int = logging.INFO) -> float:
    """
    Convert a server-side ee.Number to a client-side float
    only if the chosen level is enabled.  Prevents accidental
    getInfo() calls when verbose logging is off.
    """
    if logger.isEnabledFor(level):
        return round(obj.getInfo(), prec)
    raise RuntimeError(
        f"ee2float called for level {level} which is below the active logger threshold"
    )

def number_agree(n: int, singular: str, plural: str | None = None, zero_word: str | None = None) -> str:

    word = singular if n == 1 else (plural or singular + "s")
    num  = str(n) if (n != 0 or zero_word is None) else zero_word
    return f"{num} {word}"

## 2.3 Setup Drive

In [ ]:
def setup_drive(
                drive_base: str,
                mount_point: str,
                ) -> Path:

    # Cache the mount step on the function itself
    if not getattr(setup_drive, "_mounted", False):
        drive.mount(mount_point, force_remount=False)
        setup_drive._mounted = True

    base = Path(drive_base)
    workdir = base if base.is_absolute() else Path(mount_point) / base
    return workdir

## 2.4 Get or Create Fall **Composite**

In [ ]:
def clear_snowfree_s2_composite(date_range, aoi_geom, cfg):

    start_date = ee.Date(date_range[0])
    end_date = ee.Date(date_range[1])

    early_start_date = start_date.advance(-1, 'month')

    def _prep_s2(img):
        scaled_optical_bands = img.select(cfg.optical_bands).divide(10000)
        return scaled_optical_bands.addBands(img.select('SCL'))

    #helper masking function
    def _mask_clear(img):
        scl  = img.select('SCL')
        clear_mask = (
            scl.eq(4)  # vegetation
            .Or(scl.eq(5))  # bare soil
            .Or(scl.eq(6))  # water
            .Or(scl.eq(11)) # snow / ice
        )
        img = img.updateMask(clear_mask)

        return img

    def _mask_lax(img):
        scl  = img.select('SCL')
        clear_mask = (
            scl.eq(1)  # saturated/defective
            .Or(scl.eq(2))  # dark area
            .Or(scl.eq(3))  # cloud shadow
            .Or(scl.eq(4)) # vegetated
            .Or(scl.eq(5))  # non vegetated
            .Or(scl.eq(6))  # water
            .Or(scl.eq(7))  # Unclassified
            #.Or(scl.eq(8))  # Cloud Medium
        )
        img = img.updateMask(clear_mask)

        return img

    base_coll = (cfg.ic
              .filterDate(start_date, end_date)
              .filterBounds(aoi_geom))

    early_coll = (cfg.ic
              .filterDate(early_start_date, end_date)
              .filterBounds(aoi_geom))

    strict_coll = (base_coll
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
                  .filter(ee.Filter.lt('SNOW_ICE_PERCENTAGE', 1))
                  .map(_prep_s2)
                   )

    lax_coll = (base_coll
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70))
                  .filter(ee.Filter.lt('SNOW_ICE_PERCENTAGE', 10))
                  .map(_prep_s2)
                   )

    early_lax_coll = (early_coll
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70))
                  .filter(ee.Filter.lt('SNOW_ICE_PERCENTAGE', 10))
                  .map(_prep_s2)
    )


    clear_image = strict_coll.map(_mask_clear).median()
    tier_2_image = strict_coll.map(_mask_lax).median()
    tier_3_image = lax_coll.map(_mask_clear).median()
    tier_4_image = lax_coll.map(_mask_lax).median()
    early_lax_image = early_lax_coll.map(_mask_lax).median()


    final_composite = (clear_image
                  .unmask(tier_2_image)      # Fill gaps with winn_2
                  .unmask(tier_3_image)      # Fill remaining gaps with winn_3
                  .unmask(tier_4_image)
                  .unmask(early_lax_image) # Fill remaining gaps with winn_early
)

    return final_composite.select(cfg.optical_bands).clip(aoi_geom)

def describe_training_data(paths: list[Path],
                           *,
                           counties_fc: ee.FeatureCollection,
                           date_col: str) -> "TrainingDataSummary":
    """
    Read the training polygons and report what they actually contain: which
    counties they sit in, and which autumns they belong to.

    Nothing about the training data is written down in this notebook. Swap the
    files for polygons from another region or another winter and the training
    composite, the tested-region warning, and the reported counts all follow.

    Counties come from each polygon's bounding rectangle rather than the
    polygon itself. A rectangle always contains the polygon inside it, so this
    can select a county that turns out not to be needed, but it can never miss
    one that is.
    """
    frames = []

    for path in paths:
        try:
            gdf = gpd.read_file(path)
        except Exception as err:
            raise ManureSpotterAbort(
                f"Could not read the training file '{path}'.\n\n"
                "Check that DRIVE_DATA_FOLDER in section 0 points at the folder "
                "holding the .gpkg files from the Zenodo download, and that "
                "their names are unchanged.\n\n"
                f"{err}"
            ) from None

        if date_col not in gdf.columns:
            raise ManureSpotterAbort(
                f"The training file '{path.name}' has no '{date_col}' column, "
                "so there is no way to tell which winter its polygons belong "
                f"to.\nColumns found: {list(gdf.columns)}"
            )

        frames.append(gdf[[date_col, "geometry"]])

    combined = pd.concat(frames, ignore_index=True)

    dates = pd.to_datetime(combined[date_col], errors="coerce")
    if dates.isna().any():
        raise ManureSpotterAbort(
            number_agree(int(dates.isna().sum()), "training polygon",
                         "training polygons")
            + f" have a '{date_col}' value that is not a date this notebook "
              "can read. Dates should look like 2025-02-15."
        )

    days = dates.dt.strftime("%Y-%m-%d")
    fall_years = sorted({_fall_year_for(d) for d in days})

    boxes = gpd.GeoDataFrame(geometry=combined.geometry.envelope,
                             crs=frames[0].crs)

    geoids = (counties_fc
              .filterBounds(geemap.gdf_to_ee(boxes, geodesic=False))
              .aggregate_array("GEOID")
              .getInfo())

    if not geoids:
        raise ManureSpotterAbort(
            "The training polygons do not fall inside any county. That usually "
            "means the training files are in an unexpected coordinate system."
        )

    return TrainingDataSummary(
        county_ids    = sorted(set(geoids)),
        fall_years    = fall_years,
        polygon_count = len(combined),
        first_date    = days.min(),
        last_date     = days.max(),
    )


def _ensure_image_collection(asset_id: str) -> None:
    """Create the collection that holds the per-county composites, once."""
    if asset_exists(asset_id):
        return

    last_error = None
    for type_name in ("IMAGE_COLLECTION", "ImageCollection"):
        try:
            ee.data.createAsset({"type": type_name}, asset_id)
            return
        except ee.ee_exception.EEException as err:
            last_error = err

    raise ManureSpotterAbort(
        f"Could not create the collection '{asset_id}' that holds the autumn "
        f"composites.\n{last_error}"
    )


def _list_collection(asset_id: str) -> dict[str, dict]:
    """Everything currently in the collection, keyed by full asset id."""
    found: dict[str, dict] = {}
    page_token = None

    while True:
        request = {"parent": asset_id}
        if page_token:
            request["pageToken"] = page_token

        response = ee.data.listAssets(request) or {}

        for entry in response.get("assets", []):
            # Key on the image's own name inside the collection ("55009_2025"),
            # which is the last path segment however the id is spelled.
            key = entry.get("id") or entry.get("name") or ""
            if key:
                found[key.rstrip("/").rsplit("/", 1)[-1]] = entry

        page_token = response.get("nextPageToken")
        if not page_token:
            return found


def get_or_build_fall_composite(
    *,
    fall_year: int,
    date_range: tuple[str, str],
    county_ids: list[str],
    county_labels: dict[str, str],
    counties_fc: ee.FeatureCollection,
    cfg: SensorConfig,
    collection: Path,
    label: str = "fall composite",
    ) -> "FallCompositeResult":
    """
    Assemble the snow-free autumn composite for the requested counties.

    One image is stored per county per autumn. A run mosaics together the ones
    it needs and starts builds for any that are missing. Every operation in
    clear_snowfree_s2_composite is per-pixel, so a mosaic of per-county images
    is identical to one image built over all of them at once — there are no
    seams to worry about.

    Storing them separately means adding a county later costs one county rather
    than rebuilding everything, and nothing has to be pasted into section 0.
    """

    def _prep_fall_comp_band_names(composite):
        return composite.select(cfg.optical_bands).rename(
                [b + '_NS' for b in cfg.optical_bands]  # NS = non snow
                )

    msgs: list[str] = []
    collection_id = str(collection)

    _ensure_image_collection(collection_id)
    existing = _list_collection(collection_id)

    def _asset_name(geoid: str) -> str:
        return f"{geoid}_{fall_year}"

    present = [g for g in county_ids if _asset_name(g) in existing]
    missing = [g for g in county_ids if _asset_name(g) not in existing]

    # ---- were the ones we already have built from this same window? --------
    stale = []
    for geoid in present:
        props = (existing[_asset_name(geoid)].get("properties") or {})
        stamped = (props.get("manure_spotter_window_start"),
                   props.get("manure_spotter_window_end"))
        if all(stamped) and tuple(stamped) != tuple(date_range):
            stale.append(f"  {county_labels.get(geoid, geoid)}: built from "
                         f"{stamped[0]} to {stamped[1]}")

    if stale:
        raise ManureSpotterAbort(
            f"Some saved {label} images were built from a different stretch of "
            f"autumn than this run needs ({date_range[0]} to "
            f"{date_range[1]}):\n\n" + "\n".join(stale) + "\n\n"
            "That happens if COMPOSITE_WINDOW changed after they were built. "
            f"Delete them from '{collection_id}' and run again to rebuild."
        )

    # ---- start builds for anything missing ---------------------------------
    tasks = []
    for geoid in missing:
        county_geom = counties_fc.filter(ee.Filter.eq("GEOID", geoid)).geometry()

        composite = clear_snowfree_s2_composite(date_range, county_geom, cfg)
        composite = composite.set({
            "manure_spotter_county":       geoid,
            "manure_spotter_fall_year":    fall_year,
            "manure_spotter_window_start": date_range[0],
            "manure_spotter_window_end":   date_range[1],
        })

        name = _asset_name(geoid)
        task = ee.batch.Export.image.toAsset(
            image       = composite,
            description = f"fall_composite_{name}",
            assetId     = f"{collection_id}/{name}",
            region      = county_geom,
            scale       = 10,
            maxPixels   = 1e13,
        )
        task.start()
        tasks.append(task)

    if present:
        msgs.append(
            f"{label} for autumn {fall_year}: "
            f"{number_agree(len(present), 'county', 'counties')} already built")

    if missing:
        msgs.append(
            f"{label} for autumn {fall_year}: started building "
            f"{number_agree(len(missing), 'county', 'counties')} — "
            + ", ".join(county_labels.get(g, g) for g in missing))
        return FallCompositeResult(None, msgs, tasks)

    # ---- everything present, so mosaic it ----------------------------------
    mosaic = ee.ImageCollection(
        [ee.Image(f"{collection_id}/{_asset_name(g)}") for g in present]
    ).mosaic()

    return FallCompositeResult(_prep_fall_comp_band_names(mosaic), msgs, [])


## 2.5 Load Prior Detections

In [ ]:
def load_newest_geojson_for_date(date_str: str,
                                  table_export_folder: Path,
                                  prefix: str) -> gpd.GeoDataFrame | None:
    """
    Find the most recent GeoJSON whose filename contains the target date token
    """
    pattern = f"{prefix}*_{date_str}.geojson"
    matches = sorted(table_export_folder.glob(pattern),
                     key=lambda p: p.stat().st_mtime,
                     reverse=True)
    if not matches:
        return None

    return gpd.read_file(matches[0])

def load_prior_detections(target_date: str,
                          table_export_folder: Path,
                          prefix: str,
                          lookback_days: int,
                          date_format: str
                          ) -> ee.FeatureCollection | None:
    if lookback_days <= 0:
        logger.info("lookback_days <= 0, not attempting to load prior detections\n")
        return None

    t_date = datetime.strptime(target_date, date_format).date()

    lookback_dates = []
    for i in range(1, lookback_days + 1):
        d_str = (t_date - timedelta(days=i)).strftime(date_format)
        lookback_dates.append(d_str)

    available_dates = []
    lookback_gdfs = []
    for date in lookback_dates:
        gdf = load_newest_geojson_for_date(date, table_export_folder, prefix)
        if gdf is not None:
            lookback_gdfs.append(gdf)
            available_dates.append(date)

    if lookback_gdfs == []:
        logger.info("no prior detection data found in %i-day lookback window \n",
                    lookback_days)
        return None

    logger.info("prior detection data found in %i-day lookback window: %s",
                 lookback_days,
                 ", ".join(available_dates)
                )


    merged_gdf = gpd.GeoDataFrame(pd.concat(lookback_gdfs, ignore_index=True))

    merged_fc = geemap.gdf_to_ee(merged_gdf, geodesic=True)

    return merged_fc

# 3.&nbsp;Check for valid imagery for target date

## 3.0 Build Image Collection

In [ ]:
def daily_ic(
              sensor_cfg: SensorConfig,
              date: ee.Date,
              geom: ee.Geometry,
              ):

    ic = sensor_cfg.ic
    return (ic
            .filterDate(date, date.advance(1, "day"))
            .filterBounds(geom))

In [ ]:
def annotate_effective_cloud(ic: ee.ImageCollection, sensor_cfg: SensorConfig) -> ee.ImageCollection:

    def _set(img):
        cloud_cover = ee.Number(img.get(sensor_cfg.cloud_pct_prop))

        if sensor_cfg.cirrus_pct_prop:
            cirrus_cover = ee.Number(img.get(sensor_cfg.cirrus_pct_prop))
            effective_cover = cloud_cover.subtract(cirrus_cover)
        else:
            effective_cover = cloud_cover

        return img.set({'EFFECTIVE_CLOUD': ee.Number(effective_cover)})

    return ic.map(_set)



In [ ]:
def log_effective_cloud_per_image(ics: dict[str, ee.ImageCollection],
                                  ) -> None:
    """Log each image's EFFECTIVE_CLOUD at DEBUG level"""
    if not logger.isEnabledFor(logging.DEBUG):
        return

    for skey, ic in ics.items():
        ic2 = ic

        count = ic2.size().getInfo()
        if count == 0:
            logger.debug("%s: no images", SENSORS[skey].pretty)
            continue

        ids  = ic2.aggregate_array('system:index').getInfo()
        vals = ic2.aggregate_array('EFFECTIVE_CLOUD').getInfo()
        for pid, val in zip(ids, vals):
            logger.debug("%s image %s — EFFECTIVE_CLOUD = %.2f%%",
                         SENSORS[skey].pretty, pid, float(val))

In [ ]:
def drop_and_log_high_cloud(
                            ics: dict[str, ee.ImageCollection],
                            sensors: dict[str, SensorConfig],
                            max_cloud: int,
                            logger: logging.Logger
                            ) -> CloudCullResult:
    filtered: dict[str, ee.ImageCollection] = {}
    for k, ic in ics.items():
        over = ic.filter(ee.Filter.greaterThanOrEquals('EFFECTIVE_CLOUD', max_cloud))
        n = over.size().getInfo()
        if n:
            ids  = over.aggregate_array('system:index').getInfo()
            vals = over.aggregate_array('EFFECTIVE_CLOUD').getInfo()
            logger.info(
                "Dropping %d %s image(s) with EFFECTIVE_CLOUD ≥ %d%%: %s",
                n, sensors[k].pretty, max_cloud,
                ", ".join(f"{i}:{round(v,2)}%" for i, v in zip(ids, vals))
            )
        filtered[k] = ic.filter(ee.Filter.lt('EFFECTIVE_CLOUD', max_cloud))

    available = [k for k, ic in filtered.items() if ic.size().getInfo() > 0]
    counts = {k: filtered[k].size().getInfo() for k in available}
    return CloudCullResult(filtered, available, counts)

## 3.1 Build Image (And Optional Export)

In [ ]:
def get_img_for_date(
                      date_str,
                      aoi_geom,
                      sensor_cfg: SensorConfig
                      ):

    start_date = ee.Date(date_str)
    end_date = start_date.advance(1, 'day')

    images = (
              sensor_cfg.ic
              .filterDate(start_date, end_date)
              .filterBounds(aoi_geom)
    )

    return (images.mosaic()
                  .select(sensor_cfg.optical_bands)
                  .multiply(sensor_cfg.scalar)
                  )

In [ ]:
def export_image(image: ee.Image,
                 add_timestamp: bool = True,
                 *,
                 aoi: ee.Geometry,
                 filename: str,
                 scale: int = 10,
                 folder = 'GEE_Exports'):

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    if add_timestamp:
        filename="_".join([filename, timestamp])

    image_export_task = ee.batch.Export.image.toDrive(
        image=image.clip(aoi),
        description=filename,
        folder= folder,
        fileNamePrefix=filename,
        region=aoi,
        scale=scale,
        maxPixels=1e13
    )
    image_export_task.start()

## 3.2 Get Sensor AOI Coverage
(intersection of sensor coverage and AOI)

In [ ]:
def get_sensor_aoi_coverage(skey: str,
                            image_collection: ee.ImageCollection,
                            aoi: AOIContext):

    def _get_ic_geom(image_collection):
        img_footprints = image_collection.map(lambda img: ee.Feature(img.geometry()))

        ic_coverage = (ee.FeatureCollection(img_footprints)
                      .union()
                      .geometry()
                      )

        ic_aoi_coverage = ic_coverage.intersection(aoi.geom, 1)

        return ic_aoi_coverage

    def _keep_only_polygons(geom):

        parts = ee.Geometry(geom).geometries()

        def _tag(g):
            g = ee.Geometry(g)
            return ee.Feature(g).set({
                'isPoly': ee.List(['Polygon', 'MultiPolygon']).contains(g.type())
            })

        fc = ee.FeatureCollection(parts.map(_tag))

        poly_fc = fc.filter(ee.Filter.eq('isPoly', True))
        return poly_fc.geometry()


    target_date_aoi_coverage = _get_ic_geom(image_collection)

    clean_geom = _keep_only_polygons(target_date_aoi_coverage)

    aoi_sensor_intersect = clean_geom.intersection(aoi.geom, maxError = 10)

    focal_mask = (ee.Image()
               .paint(aoi_sensor_intersect, 1)   # burn 1s inside the geometry
               .selfMask()

               )
    acres = aoi_sensor_intersect.area(maxError=1000).multiply(aoi.m2acres)

    return SensorCoverage(
      date = aoi.date_str,
      sensor = skey,
      geom = aoi_sensor_intersect,
      mask = focal_mask,
      acres = acres,
      percent_aoi = acres.divide(aoi.acres).multiply(100)
      )


## 3.3 Check for Clouds

In [ ]:
def _clean_and_analyze_cloud_mask(cloud_mask: ee.Image,
                                  coverage: SensorCoverage,
                                  params: ClearskyMaskParams,
                                  qa_mosaic = False):

    kernel = params.buffer_kernel

    # reprojecting saves us thousands of eecu-seconds
    coarse = cloud_mask.reproject(cloud_mask.projection(), None, params.coarse_scale)
    coarse_dilated   = coarse.focal_max(
        kernel.size,
        kernel.shape,
        kernel.units
        )

    inverse_mask = (coarse_dilated
                .unmask()
                .Not()
                .selfMask()
                .updateMask(coverage.mask)
                )

    if qa_mosaic:
        inverse_mask = inverse_mask.updateMask(qa_mosaic.mask())

    inv_cluster_size = (inverse_mask
                        .connectedPixelCount(maxSize=params.cluster_thresh,
                                            eightConnected=False))

    inverse_clean = inverse_mask.updateMask(inv_cluster_size.gte(params.cluster_thresh))

    # converting to and from vectors saves compute
    inv_vec = (inverse_clean
           .selfMask()
           .reduceToVectors(
               geometry       = coverage.geom,
               scale          = params.coarse_scale,
               geometryType   = 'polygon',
               eightConnected = False,
               labelProperty  = 'value',
               maxPixels      = 1e13))

    clearsky_mask = (ee.Image()
                .paint(inv_vec, 1)
                .rename('mask')
                .updateMask(coverage.mask)
                .reproject(inverse_clean.projection()))  # ensure 10 m grid

    clearsky_mask_acres = inv_vec.geometry().area(maxError=10_000).multiply(coverage.m2acres)

    return clearsky_mask, clearsky_mask_acres


# is the reliance on separate focal_mask and focal_geom redundant
def s2_build_buffered_cloud_mask(image_collection: ee.ImageCollection,
                                 coverage: SensorCoverage,
                                 params: ClearskyMaskParams):

    scl_only = image_collection.select('SCL')

    scl_mosaic = (scl_only
                  .mosaic()
                  .updateMask(coverage.mask)
                  )

    cloud_classes = [3, 7, 9] # shadow, unclassified, high (not touching medium)


    cloud_mask   = (scl_mosaic
                    .remap(cloud_classes, [1] * len(cloud_classes), 0)
                    .selfMask())

    clearsky_mask, clearsky_mask_acres = _clean_and_analyze_cloud_mask(
                                  cloud_mask = cloud_mask,
                                  coverage = coverage,
                                  params = params
                                  )


    return clearsky_mask, clearsky_mask_acres




def l89_build_buffered_cloud_mask(  image_collection: ee.ImageCollection,
                                    coverage: SensorCoverage,
                                    params: ClearskyMaskParams):
    # bit helpers
    def _bit(img, bit):
        return img.bitwiseAnd(1 << bit).neq(0)

    def _conf(img, start_bit):
        return img.rightShift(start_bit).bitwiseAnd(3)

    qa_only = image_collection.select('QA_PIXEL')

    qa_mosaic = (qa_only
              .mosaic()
              .updateMask(coverage.mask)
              )

    core_cloud = _bit(qa_mosaic, 3) # might be unnecessary to use both of these
    high_conf = _conf(qa_mosaic, 8).eq(3)

    cloud_flag = core_cloud.And(high_conf)
    cloud_mask = cloud_flag.selfMask()

    clearsky_mask, clearsky_mask_acres = _clean_and_analyze_cloud_mask(
                                  cloud_mask = cloud_mask,
                                  coverage = coverage,
                                  params = params,
                                  qa_mosaic = qa_mosaic
                                  )


    return clearsky_mask, clearsky_mask_acres

def build_clearsky_mask( ic: ee.ImageCollection,
                      sensor_aoi_coverage: SensorCoverage,
                      params: ClearskyMaskParams):

    if sensor_aoi_coverage.sensor == "s2":
        clearsky_mask, clearsky_mask_acres = s2_build_buffered_cloud_mask(
                                      image_collection = ic,
                                      coverage = sensor_aoi_coverage,
                                      params = params
                                      )

    elif sensor_aoi_coverage.sensor == "l89":
        clearsky_mask, clearsky_mask_acres = l89_build_buffered_cloud_mask(
                                      image_collection = ic,
                                      coverage = sensor_aoi_coverage,
                                      params = params
                            )

    return ClearskyMaskResult(
        mask = clearsky_mask,
        acres = clearsky_mask_acres,
        coverage = sensor_aoi_coverage,
        params = params
    )

## 3.4 Cropland Mask

In [ ]:
# before inspecting for snow, we want to crop to just croplands for the date in question
# this masks out wc_2021 non-crop pixels AND tiger 2016 roads

def build_cropland_mask(aoi_geom):

    wc_2021 = ee.Image('ESA/WorldCover/v200/2021')

    # Remap: 40 to 1 (cropland), everything else to 0
    cropland_mask = (
        wc_2021
          .remap(
              [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
              [ 0,  0,  0,  1,  0,  0,  0,  0,  0,  0,   0]
          )
          .rename('cropland')
          .selfMask()
          )

    roads_fc   = ee.FeatureCollection('TIGER/2016/Roads')
    roads_aoi = roads_fc.filterBounds(aoi_geom)
    roads_aoi_buff = roads_aoi.map(lambda f: f.buffer(CFG.road_buffer).set({'burn':1}))

    non_roads_mask = (ee.Image(1)
                  .byte()
                  .paint(roads_aoi_buff, 0)
                  .selfMask())

    cropland_mask_img = cropland_mask.And(non_roads_mask)

    return cropland_mask_img

## 3.5 Check for Snow

In [ ]:

def calculate_vis_brightness(img: ee.Image,
                             params: VisBrightnessParams):
    b = img.select
    band_map = params.band_mapping
    blue = b(band_map["blue"])
    green = b(band_map["green"])
    red = b(band_map["red"])
    return blue.add(green).add(red).divide(3).rename('VIS_BRIGHT')

def vis_brightness_by_tract(image: ee.Image,
                            mask: ee.Image,
                            aoi: SensorCoverage,
                            params: VisBrightnessParams):

    nbsi_img = (calculate_vis_brightness(image, params)
            .clip(aoi.geom)
            .updateMask(mask)
            .reproject(
                crs   = image.select(params.band_mapping["blue"]).projection(),
                scale = params.coarse_scale))

    geo_blocks = params.geo_blocks.filterBounds(aoi.geom)

    stats = nbsi_img.reduceRegions(
          collection = geo_blocks,
          reducer    = ee.Reducer.mean(),
          scale      = params.coarse_scale
          )

    return nbsi_img, stats

def get_vis_brightness_outputs(img: ee.Image,
                              mask: ee.Image,
                              aoi: SensorCoverage,
                              params: VisBrightnessParams):

    brightness_img, stats = vis_brightness_by_tract(img, mask, aoi, params)
    high_fc   = stats.filter(ee.Filter.gt("mean", params.vis_brightness_thresh))
    high_mask = ee.Image().paint(high_fc.union().geometry(), 1).selfMask()

    high_nbsi_acres = (high_fc.geometry()
                .intersection(aoi.geom, 1)
                .area(maxError=10_000)
                .multiply(aoi.m2acres)
                )

    return VisBrightnessOutput(
                  img=brightness_img,
                  stats=stats,
                  high_fc= high_fc,
                  high_mask=high_mask,
                  pct_high=high_nbsi_acres.divide(aoi.acres).multiply(100),
                  high_acres=high_nbsi_acres
                  )

#4.&nbsp; Load or Train Classifier

## 4.1 Load Classifier

In [ ]:
def asset_exists(asset_id: str) -> bool:
    try:
        ee.data.getAsset(asset_id)
        return True
    except ee.ee_exception.EEException:
        return False

## 4.2 Load Training Data

In [ ]:
def find_duplicate_ids(gdf: gpd.GeoDataFrame,
                       *,
                       id_col: str ):

    counts = gdf[id_col].value_counts()        # how many times each id appears
    dup_ids = counts[counts > 1].index.tolist()    # only those >1
    dup_rows = gdf[gdf[id_col].isin(dup_ids)].sort_values(id_col)

    return DuplicateIDsReport(dup_ids, dup_rows)

def _read_gdf(path: Path) -> gpd.GeoDataFrame:
    try:
        gdf = gpd.read_file(path)
    except Exception as e:
        raise IOError(f"Failed to read vector data from {path}") from e
    logger.debug("Loaded %s: %d rows, CRS=%s", path.name, len(gdf), getattr(gdf.crs, "to_string", lambda: gdf.crs)())
    return gdf

def _raise_if_invalid_schema( gdf: gpd.GeoDataFrame,
                              *,
                              params: DataImportConfig):
    id_col = params.id_col
    date_col = params.date_col

    # Every training file carries a date. Only the ground-truth files carry an
    # 'id', so its uniqueness is checked where the column exists rather than
    # being required everywhere.
    if date_col not in gdf.columns:
        raise ValueError(
            f"'{date_col}' column not found in {list(gdf.columns)}."
        )

    if id_col in gdf.columns and params.enforce_unique_ids:
        report = find_duplicate_ids(gdf, id_col = id_col)
        if not report.is_empty():
            raise DuplicateIDsError(f"'{id_col}' values are not unique", report)

def path_to_ee_fc(path: Path,
                  *,
                  params: DataImportConfig):

    gdf = _read_gdf(path)

    if params.copy_gdf:
        gdf = gdf.copy()

    _raise_if_invalid_schema(gdf, params = params)

    gdf[params.date_col] = gdf[params.date_col].astype(str)

    fc = geemap.gdf_to_ee(gdf, geodesic=params.geodesic)

    return fc

def safe_path_to_ee_fc(path: Path,
                       *,
                       params: DataImportConfig):
    try:
        return path_to_ee_fc(path, params = params)
    except DuplicateIDsError as e:
        print(f"Duplicate IDs (n={len(e.report.ids)}): {e.report.ids[:10]}",
              "…" if len(e.report.ids) > 10 else "")
        try:
            from IPython.display import display
            display(e.report.rows.head(20))
        finally:
            raise

In [ ]:
def add_binary_class_from_label(fc: ee.FeatureCollection,
                                *,
                                params: DataImportConfig)->ee.FeatureCollection:
    manure_label = params.manure_label
    non_manure_label = params.non_manure_label
    type_prop = params.type_col
    class_prop = params.class_col

    allowed_types = ee.List([manure_label, non_manure_label])

    # look for bad labels
    bad_types = fc.filter(
        ee.Filter.Not(ee.Filter.inList(type_prop, allowed_types))
    )

    if bad_types.size().getInfo() > 0:          # client-side check
        bad_vals = bad_types.aggregate_array(type_prop).distinct().getInfo()
        raise ValueError(
            f"Unexpected '{type_prop}' values found: {bad_vals}"
        )

    # map the two legal strings to 1 and 0 classes
    def _add_class(feat):
        is_manure = ee.String(feat.get(type_prop)).equals(manure_label) #boolean

        class_num = ee.Algorithms.If(is_manure, ee.Number(1), ee.Number(0))
        return feat.set(class_prop, class_num)

    return fc.map(_add_class)

## 4.3 Build Samples

In [ ]:
def build_training_img_dict(
                            training_polys,
                            sensor_cfg: SensorConfig
                            ):

    # fetch all the different daily geometries for the training data
    training_days = training_polys.aggregate_array('use_date').distinct().getInfo()

    day_geoms = {
        day: (training_polys
                .filter(ee.Filter.eq('use_date', day))
                .union()        # single feature
                .geometry())    # convert to Geometry
        for day in training_days
    }


    training_images = {
        day: get_img_for_date(day, day_geoms[day], sensor_cfg)
                        for day in training_days}

    return training_images


In [ ]:
def sample_regions_for_date(date_str, img, training_polys, training_fall_composite):

    filtered_polys = training_polys.filter(ee.Filter.eq('use_date', date_str))

    img_stack = img.addBands(training_fall_composite)

    sampled_pix = (img_stack
                  .sampleRegions(
                      collection=filtered_polys,
                      properties=['class'],
                      scale=10)
                  )


    return sampled_pix

def create_flat_sample(training_images, training_polys, training_fall_composite):
    sample_complete = [sample_regions_for_date(
                                                date,
                                                training_images[date],
                                                training_polys,
                                                training_fall_composite)
                        for date in training_images.keys()
                      ]

    return ee.FeatureCollection(sample_complete).flatten()

## 4.4 Train Random Forest Classifer

In [ ]:
def train_rf_classifier(sample_fc: ee.FeatureCollection,
                        bands: list[str],
                        params: RFParams,
                        class_prop: str = 'class',
                        ):

    classifier = ee.Classifier.smileRandomForest(
        numberOfTrees=params.num_trees,
        variablesPerSplit=params.vars_per_split,
        minLeafPopulation=params.min_leaf_pop,
        bagFraction=params.bag_frac,
        maxNodes=params.max_nodes,
        seed=params.seed
    ).train(
        features=sample_fc,
        classProperty=class_prop,
        inputProperties=bands
    )

    return classifier

In [ ]:
def save_classifier(classifier, gee_asset_root, poly_count, sensor):

    datestamp = datetime.now().strftime('%Y%m%d')

    asset_path = gee_asset_root / f'rf_{sensor}_{poly_count}_polys_{datestamp}'

    task_rf_export = ee.batch.Export.classifier.toAsset(
        classifier = classifier,
        description = f'export_rf_{sensor}_{poly_count}',
        assetId     = str(asset_path)
    )

    task_rf_export.start()

    return asset_path

#5.&nbsp;Inference on Target Date Imagery
no helpers


#6.&nbsp;Post Processing

## 6.1 Convert raw classifier output to cleanup-up features

In [ ]:
def drop_clusters_below_thresh( two_class_img: ee.Image,
                                cluster_thresh: int
                 ):
    class_1_mask = two_class_img.selfMask()

    cluster_size = class_1_mask.connectedPixelCount(
        maxSize=cluster_thresh, eightConnected=True)

    cleaned = class_1_mask.updateMask(cluster_size.gte(cluster_thresh))
    return cleaned

In [ ]:
def merge_and_clean_clusters(two_class_img: ee.Image,
                             kernel: Kernel,
                             cluster_thresh: int,
                             aoi: ee.Geometry
                             ):
    ## Dilation
    closed = (two_class_img.unmask(0)
                .focal_max(kernel.size + 20, kernel.shape, kernel.units)
                .focal_min(20,               kernel.shape, kernel.units)
    )

    ## Drop small clusters
    closed_cleaned = drop_clusters_below_thresh(closed, cluster_thresh)

    ## Convert to vectors

    cluster_polys = (
        closed_cleaned
          .selfMask()
          .reduceToVectors(
              reducer        = ee.Reducer.countEvery().setOutputs(['pixel_count']), #test!!
              geometry       = aoi,  # or any AOI
              scale          = 10,
              geometryType   = 'polygon',
              eightConnected = True,
              labelProperty  = None,
              maxPixels      = 1e13
                  )
                )
    return cluster_polys

## 6.2 Merge S2 and L89 detections from same day

In [ ]:
def merge_overlapping_features(*feature_collections: ee.FeatureCollection,
                               params: FeatureExportParams) -> ee.FeatureCollection:
    """
    Merges any features that overlap or touch. Accepts an arbitrary number of
    feature collections, and returns a single feature collection. Metadata will
    be stripped
    """
    sensor_property = params.sensor_prop_name
    delimiter = params.sensor_prop_delimiter

    if not feature_collections:
      raise ValueError("Provide at least one FeatureCollection.")

    combined_fc = feature_collections[0]
    for fc in feature_collections[1:]:
        combined_fc = combined_fc.merge(fc)


    dissolved_geom = combined_fc.geometry().dissolve(maxError=1)

    parts_geoms = dissolved_geom.geometries()

    parts_fc = ee.FeatureCollection(parts_geoms.map(lambda g: ee.Feature(ee.Geometry(g))))


    # add back metadata
    matches_key = "intersecting_spreads" # GEE argument for join.saveAll, creates property

    # first find features in combined_fc intersecting features in parts_fc
    spatial_filter = ee.Filter.intersects(leftField='.geo', rightField='.geo')

    saveall_join = ee.Join.saveAll(matchesKey=matches_key)

    parts_with_source_matches = ee.FeatureCollection(
        saveall_join.apply(parts_fc, combined_fc, spatial_filter)
        )

    # second create a sensor tag
    def _create_sensor_tag(f):
        f = ee.Feature(f)
        matching_feats = ee.List(f.get(matches_key))

        matching_sensors = ee.List(matching_feats
                                   .map(lambda m: ee.Feature(m).get(sensor_property))
                                   .distinct()
                                   .sort())


        sensor_label = ee.List(matching_sensors).join(delimiter)


        return f.set(sensor_property, sensor_label)

    labeled_fc = parts_with_source_matches.map(_create_sensor_tag)

    return labeled_fc

## 6.3 Clean up detection metadata

In [ ]:
def make_feature_enricher(*,
                          params: FeatureExportParams,
                          aoi_context: AOIContext):

    def _add_attrs(feature):

        geom   = feature.geometry()
        centroid    = geom.centroid(10)
        coords      = centroid.coordinates()

        acres       = geom.area(10).multiply(aoi_context.m2acres)
        lon         = ee.Number(coords.get(0))
        lat         = ee.Number(coords.get(1))
        county_name = ee.Feature(aoi_context.counties
                                .filterBounds(centroid)
                                .first()
                                .get('NAME'))

        return feature.set({
            params.acres_prop_name    : acres,
            params.lat_prop_name      : lat,
            params.lon_prop_name      : lon,
            params.county_prop_name   : county_name
        })
    return _add_attrs

In [ ]:
def set_global_ids(fc: ee.FeatureCollection,
                    params: FeatureExportParams) -> ee.FeatureCollection:
    """
    Make nice IDs like s2_01282025_0034 and set them as `id` properties.
    """
    #sort east/west then north/south
    fc_sorted = fc.sort(params.lat_prop_name).sort(params.lon_prop_name)

    n = fc_sorted.size()
    sorted_list = fc_sorted.toList(n)
    idx = ee.List.sequence(1, n)

    zeros = '0' * params.pad

    def _mk(i):
        i = ee.Number(i)
        f = ee.Feature(sorted_list.get(i.subtract(1)))

        county_name = ee.String(f.get(params.county_prop_name)).toLowerCase().split('[^a-z0-9]+').join('')
        sensor_name = ee.String(f.get(params.sensor_prop_name))
        date_str    = ee.String(f.get(params.date_prop_name)).split('-').join('')
        num_suffix  = ee.String(zeros).cat(i.format('%.0f')).slice(-params.pad)

        pretty_id   = county_name.cat('_').cat(date_str).cat('_').cat(num_suffix).cat('_').cat(sensor_name)
        return f.set(params.id_prop_name, pretty_id)

    return ee.FeatureCollection(idx.map(_mk))

# 7.&nbsp;Image Chip Export

## 7.1 Image Export

In [ ]:
# helper that returns an RGB image visualised with per-image stretch

def dynamic_rgb_vis(img: ee.Image,
                    *,
                    aoi: ee.Geometry,
                    export_params: FeatureExportParams,
                    sensor_config: SensorConfig
                    ) -> ee.Image:
    """
    Compute per-band percentile stretch (e.g. p2–p98) for the given image
    over aoi, then visualise with gamma adjustment.
    """

    bands = [sensor_config.band_lookup[b] for b in export_params.bands]
    pct_low = export_params.pct_low
    pct_high = export_params.pct_high

    # Get per-band percentiles (returns server-side ee.Dictionary)
    percentiles = img.select(bands).reduceRegion(
        reducer=ee.Reducer.percentile([pct_low, pct_high]),
        geometry=aoi,
        scale=sensor_config.rgb_scale,
        bestEffort=True,
        maxPixels=1e8
    )

    # build per-band min/max lists
    mins = [ee.Number(percentiles.get(f'{b}_p{pct_low}'))  for b in bands]
    maxs = [ee.Number(percentiles.get(f'{b}_p{pct_high}')) for b in bands]

    # visualise with those dynamic limits and a gamma lift
    return img.visualize(
        bands=bands,
        min=mins,
        max=maxs,
        gamma=[export_params.gamma] * len(bands)
    )

def export_one(feature: ee.Feature,
               *,
               image_collection: ee.ImageCollection,
               export_params: FeatureExportParams,
               sensor_config: SensorConfig,
               filename_prefix: str):

    collection_size = image_collection.size().getInfo()

    if collection_size == 0:
        logger.warning("the image collection is empty for %s", # fix to include sensor
                       sensor_config.pretty)
        return

    geom = feature.geometry()
    buffered_geom = geom.buffer(export_params.buffer)
    bbox = buffered_geom.bounds()
    centroid = geom.centroid()
    bands = [sensor_config.band_lookup[b] for b in export_params.bands]

    scene_image = image_collection.filterBounds(geom).first().select(bands)

    scene_image_upsampled = (scene_image
            .resample(export_params.interpolation)
            .reproject(crs=export_params.crs, scale=sensor_config.export_scale))

    chip_geom  = centroid.buffer(export_params.dimension).bounds()

    chip = dynamic_rgb_vis(scene_image_upsampled,
                              aoi = chip_geom,
                              export_params = export_params,
                              sensor_config = sensor_config)


    detection_bbox = (ee.Image()
                          .paint(bbox, 0, export_params.bbox_thickness)
                          .visualize(palette=[export_params.bbox_color])
                      )
    chip_with_bbox = chip.blend(detection_bbox)


    export_image(chip_with_bbox,
           add_timestamp = False,
           aoi = chip_geom,
           filename = filename_prefix,
           scale = sensor_config.export_scale,
           folder = export_params.export_folder)

def export_all(fc: ee.FeatureCollection,
                *,
               image_collection: ee.ImageCollection,
               export_params: FeatureExportParams,
               sensor_config: SensorConfig):

    """export RGB chips with bboxes for every feature in a collection."""

    id_field = export_params.id_prop_name

    ## just for testing, delete
    n = fc.size().getInfo()
    print(f"Features in collection: {n}")

    # Pull IDs once
    ids = fc.aggregate_array(id_field).getInfo()

    ## Just for testing
    print(f"IDs fetched: {len(ids)}")

    # this will need to be map function instead. think lamba should do the trick
    # Launch exports
    for pid in ids:
        feat = ee.Feature(fc.filter(ee.Filter.eq(id_field, pid)).first())
        export_one(feat,
                   image_collection=image_collection,
                   export_params= export_params,
                   sensor_config=sensor_config,
                   filename_prefix=str(pid))

## 7.2 Wait for GEE

In [ ]:
def wait_for_detection_geojson(geojson_name: str,
                                *,
                               folder_path: Path,
                               poll_seconds: int,
                               timeout_minutes: int,
                               logger=None) -> Path | None:

    start = time.time()
    file_name = f"{geojson_name}.geojson"

    if logger:
        logger.info("Watching %s for '%s' (poll=%ss, timeout=%smin)",
                    folder_path, file_name, poll_seconds, timeout_minutes)

    while True:
        matches = sorted(folder_path.glob(file_name), key=lambda p: p.stat().st_mtime, reverse=True)
        if matches:
            if logger:
                logger.info("Detected file: %s", matches[0].name)
            return matches[0]

        if time.time() - start > timeout_minutes * 60:
            if logger:
                logger.error("Timed out waiting for %s in %s", file_name, folder_path)
            return None

        if logger:
            logger.info("Not found yet. Checking again in %s seconds...", poll_seconds)
        time.sleep(poll_seconds)

# Main Script

### Orchestrator

In [ ]:
# as configured, aoi_fc and counties point to the same variable, but in cases
# where aoi is not defined by counties, they will be different
aoi = make_aoi_context( AOI_FC,
                        date_str = CFG.target_date,
                        m2acres = CFG.m2acres,
                        counties = CFG.counties_all)

datestamp = datetime.now().strftime('%Y%m%d')
geojson_name = None

if logger.isEnabledFor(PROGRESS_LEVEL):
    progress(
        "User’s area of interest (AOI) is %.2f million acres",
        ee2float(aoi.acres.divide(1_000_000), level=PROGRESS_LEVEL),
    )

logger.info("searching for satellite images intersecting user's AOI...\n")

# Check for satellite coverage for target date, add "effective_cloud" attribute
ics = {
    skey: annotate_effective_cloud(
        daily_ic(SENSORS[skey], aoi.ee_date, aoi.geom),
        SENSORS[skey]
        )
    for skey in SENSORS
}

if logger.isEnabledFor(PROGRESS_LEVEL):
    raw_counts = {k: ics[k].size().getInfo() for k in ics}

    progress(
        "Raw scenes intersecting AOI on %s: %s",
        aoi.date_str,
        ", ".join(f"{SENSORS[k].pretty}={raw_counts[k]}" for k in raw_counts)
    )

log_effective_cloud_per_image(ics)

cull = drop_and_log_high_cloud(ics, SENSORS, CFG.max_cloud_cover,logger)
ics = cull.filtered
available = cull.available
acceptable_img_counts = cull.counts

if not available:
    raise ManureSpotterAbort(
        f"No usable satellite imagery covers your counties on {aoi.date_str}.\n"
        "Nothing went wrong. Either neither satellite passed overhead that day, "
        "or every scene was too cloudy to use.\n"
        "Try a different date."
    )


logger.info(
    "AOI imagery available for %s on %s \n",
    ", ".join(
        f"{SENSORS[k].pretty} ({number_agree(acceptable_img_counts[k], 'scene')})"
        for k in available
    ),
    aoi.date_str
)

###########################################
# 2.3 Setup Deive
###########################################

workdir = setup_drive(
                      drive_base = CFG.drive_base,
                      mount_point = CFG.mount_point)

logger.info("working directory resolved to '%s'", workdir)

CFG.ensure_table_export_path()

###########################################
# 2.3b What the training data covers
###########################################
# Everything the notebook needs to know about the training data is read from
# the files: which counties the polygons sit in, and which autumns they belong
# to. Nothing is written down here.

TRAINING_DATA = describe_training_data(
    [CFG.resolve(p) for cfg_ in SENSORS.values()
                    for p in (cfg_.paths.gt, cfg_.paths.so, cfg_.paths.sm)],
    counties_fc = CFG.counties_all,
    date_col    = CFG.data_import_params.date_col)

TRAINING_COUNTY_IDS = TRAINING_DATA.county_ids

logger.info(
    "training data: %i polygons from %s to %s, covering %s across %s",
    TRAINING_DATA.polygon_count,
    TRAINING_DATA.first_date,
    TRAINING_DATA.last_date,
    number_agree(len(TRAINING_COUNTY_IDS), "county", "counties"),
    number_agree(len(TRAINING_DATA.fall_years), "winter", "winters"))

_untested = [c for c in FOCAL_COUNTY_IDS if c not in set(TRAINING_COUNTY_IDS)]

if _untested:
    logger.warning(
        "%s of your %s counties (%s) are outside the region where this "
        "classifier was validated",
        len(_untested), len(FOCAL_COUNTY_IDS),
        ", ".join(_COUNTY_LABELS.get(c, c) for c in _untested))

###########################################
# 2.4 Fall Composites
###########################################
# The autumn composites are stored one image per county, in a collection in
# your own project. A run mosaics the counties it needs and starts builds for
# any that are missing.
#
# The two composites cover different ground on purpose. The target composite
# follows FOCAL_COUNTIES, because that is where detection happens. The training
# composite follows the counties the training polygons sit in, because that is
# where the classifier is sampled — narrowing FOCAL_COUNTIES must not narrow
# the data the classifier learns from.

_needs_training = any(SENSORS[skey].paths.rf is None for skey in SENSORS)

_composite_jobs = [("target",
                    _fall_year_for(CFG.target_date),
                    CFG.target_fall_comp_rng,
                    FOCAL_COUNTY_IDS)]

if _needs_training:
    # One training composite, so one winter. When the training data grows to
    # cover several winters this has to become one composite per winter, with
    # each polygon sampled against its own — until then, stop rather than
    # quietly sample one winter's polygons against another winter's imagery.
    if len(TRAINING_DATA.fall_years) > 1:
        raise ManureSpotterAbort(
            "The training data spans "
            + number_agree(len(TRAINING_DATA.fall_years), "winter", "winters")
            + " ("
            + ", ".join(f"autumn {y}" for y in TRAINING_DATA.fall_years)
            + "), but the notebook builds a single training composite.\n\n"
            "Sampling polygons from one winter against another winter's "
            "reference imagery would train the classifier on the wrong "
            "ground, so the run stops here instead of doing that quietly.\n\n"
            "Train on one winter at a time until per-winter training "
            "composites are supported."
        )

    _training_fall_year = TRAINING_DATA.fall_years[0]

    _composite_jobs.append(("training",
                            _training_fall_year,
                            _fall_window(_training_fall_year),
                            TRAINING_COUNTY_IDS))
else:
    logger.info("both classifiers are already trained, so no training "
                "composite is needed")

fall_composites = {}

for _label, _year, _date_rng, _counties in _composite_jobs:

    fall_composites[_label] = get_or_build_fall_composite(
                                    fall_year     = _year,
                                    date_range    = _date_rng,
                                    county_ids    = _counties,
                                    county_labels = _COUNTY_LABELS,
                                    counties_fc   = CFG.counties_all,
                                    cfg           = SENSORS["s2"],
                                    collection    = CFG.fall_composite_path,
                                    label         = f"{_label} composite")

    for line in fall_composites[_label].messages:
        logger.info(line)

_pending = sum(len(v.tasks) for v in fall_composites.values())

if _pending:
    raise ManureSpotterAbort(
        f"Started {number_agree(_pending, 'build', 'builds')} of autumn "
        "composite imagery for counties that did not have one yet.\n\n"
        "They run in your Earth Engine account in the background. Watch them "
        "in the Tasks tab at https://code.earthengine.google.com/.\n\n"
        "When they have all finished, run the notebook again — there is "
        "nothing to paste into section 0. Counties already built are not "
        "rebuilt, so adding a county later only costs that one county."
    )

target_fall_comp_img   = fall_composites["target"].img
training_fall_comp_img = (fall_composites["training"].img
                          if _needs_training else None)

###########################################
# 2.5 Load Prior Detections
###########################################

prior_detections = load_prior_detections(CFG.target_date,
                                          CFG.table_export_path,
                                          CFG.export_prefix,
                                          CFG.post_detection_suppression_days,
                                          CFG.date_format.py_iso)

## if there are prior detections, convert to raster mask
if prior_detections is not None:
    prior_detections_geom = prior_detections.geometry().dissolve(maxError=1)

    no_prior_detections_mask = (
                          ee.Image().byte().paint(prior_detections_geom, 1)
                          .unmask(0)
                          .eq(0)
                          .rename('no_prior_detection_mask')
                            )

###########################################
# 3 Preprocess Target Date Imagery
###########################################

sensor_outputs: dict[str, SensorOutputs] = {}

for skey in available:
    sensor_cfg = SENSORS[skey]
    sensor_txt = sensor_cfg.pretty

    logger.info("processing %s imagery...", sensor_txt)

    sensor_ic = ics[skey]

    ########################################
    # 3.2 raw coverage (no secondary helpers)
    ########################################

    logger.info("evaluating %s coverage of AOI on %s...",
                sensor_txt,
                aoi.date_str)

    sensor_aoi_coverage = get_sensor_aoi_coverage(skey, sensor_ic, aoi)

    if logger.isEnabledFor(PROGRESS_LEVEL):
      progress(
              "%s imagery is available for %.2f million acres (%.2f%%) "
              "of user's AOI on %s \n",
              sensor_txt,
              ee2float(sensor_aoi_coverage.acres.divide(1_000_000), level=PROGRESS_LEVEL),
              ee2float(sensor_aoi_coverage.percent_aoi, level=PROGRESS_LEVEL),
              aoi.date_str
      )


    logger.info("evaluating cloud conditions for imagery...")

    ########################################
    # 3.3 clouds
    ########################################
    clearsky_mask_params = ClearskyMaskParams(
        sensor_pretty=sensor_txt,
        coarse_scale=CFG.coarse_scale,
        buffer_kernel=sensor_cfg.cloud_kernel,
        cluster_thresh=sensor_cfg.clear_cluster,
        )

    clearsky = build_clearsky_mask(
                                    sensor_ic,
                                    sensor_aoi_coverage,
                                    clearsky_mask_params)


    if logger.isEnabledFor(PROGRESS_LEVEL):
      progress(
          "%.2f%% of the AOI %s imagery (%.2f million acres) is clear sky for %s \n",
          ee2float(clearsky.clear_pct_of_coverage),
          sensor_txt,
          ee2float(clearsky.acres.divide(1_000_000)),
          aoi.date_str
          )

    logger.info("evaluating snow cover in clear sky imagery...")

    ########################################
    # 3.4 cropland + cloud overlap
    ########################################

    crop_mask          = build_cropland_mask(sensor_aoi_coverage.geom)
    clear_crop_mask    = (clearsky.mask).And(crop_mask).And(sensor_aoi_coverage.mask)


    ########################################
    # 3.5 NBSI / snow
    ########################################

    mosaic            = sensor_ic.mosaic().multiply(sensor_cfg.scalar)

    brightness_params = VisBrightnessParams(
                            band_mapping=sensor_cfg.band_lookup,
                            vis_brightness_thresh=sensor_cfg.vis_brightness_thresh,
                            coarse_scale=CFG.coarse_scale,
                            geo_blocks=ee.FeatureCollection(CFG.geo_blocks)
                            )

    vis_brightness = get_vis_brightness_outputs(
                        mosaic,
                        clear_crop_mask,
                        sensor_aoi_coverage,
                        brightness_params)

    if logger.isEnabledFor(PROGRESS_LEVEL):

      pct_high_nbsi = ee2float(vis_brightness.pct_high)

      progress(
          "%.2f%% of %s AOI imagery (%.2f million acres) is snowy (NBSI > %.2f) on %s \n",
          pct_high_nbsi,
          sensor_txt,
          ee2float(vis_brightness.high_acres.divide(1_000_000)),
          sensor_cfg.vis_brightness_thresh,
          aoi.date_str
          )

      if pct_high_nbsi < CFG.min_masked_pct:
        progress("Skipping %s on %s: snow cover less than min snow threshold (%.2f%%). \n",
                  sensor_txt, aoi.date_str, CFG.min_masked_pct)

        artifact = SensorOutputs(
            ic = sensor_ic,
            aoi_coverage = sensor_aoi_coverage,
            clearsky = clearsky,
            crop_mask = crop_mask,
            vis_brightness = vis_brightness,
            inference_bands = None,
            target_image = None,
            rf_output = None,
            noise_removed = None,
            detection_polys = None
            )

        sensor_outputs[skey] = artifact

        continue

    clear_crop_snow_mask = clear_crop_mask.And(vis_brightness.high_mask)


    ###########################################
    # 4 Load or Train Classifier
    ###########################################

    #### 4.1 Load Classifier (if possible) ####
    classifier = None

    rf_path = sensor_cfg.paths.rf

    if rf_path and asset_exists(str(rf_path)):
        logger.info('%s classifier already exists. Loading classifier %s... \n',
                    sensor_txt,
                    rf_path.name)

        classifier = ee.Classifier.load(str(rf_path))



    elif rf_path and not asset_exists(str(rf_path)):
        raise ManureSpotterAbort(
            f"Could not open the {sensor_txt} classifier '{rf_path}'.\n"
            "Either the name is wrong, or the asset belongs to a different "
            "Earth Engine project and has not been shared with you.\n"
            f"Fix the name in section 0, or clear that setting to train a new "
            "classifier from your training data."
        )

    else:
        logger.info('no %s classifier indicated. training new classifier '
                    'attempting to import training data...',
                    sensor_txt)

        #### 4.2 Load Training Data and Add Binary Class Labels###########
        logger.info('attempting to upload %s training data to Google Earth Engine',
                    sensor_txt)

        groundtruth_fc = safe_path_to_ee_fc(CFG.resolve(sensor_cfg.paths.gt),
                                            params = CFG.data_import_params)
        sat_nonmanure_fc = safe_path_to_ee_fc(CFG.resolve(sensor_cfg.paths.so),
                                            params = CFG.data_import_params)
        sat_manure_fc = safe_path_to_ee_fc(CFG.resolve(sensor_cfg.paths.sm),
                                           params = CFG.data_import_params)

        groundtruth_fc = add_binary_class_from_label(groundtruth_fc, params = CFG.data_import_params)

        # sat_nonmanure is all non-manure or all manure, so easy
        sat_nonmanure_fc = sat_nonmanure_fc.map(lambda f: f.set('class', 0))
        sat_manure_fc = sat_manure_fc.map(lambda f: f.set('class', 1))

        training_polys = groundtruth_fc.merge(sat_manure_fc).merge(sat_nonmanure_fc)


        count_class_0 = training_polys.filter(ee.Filter.eq('class', 0)).size().getInfo()
        count_class_1 = training_polys.filter(ee.Filter.eq('class', 1)).size().getInfo()

        logger.info("Number of %s features with class 0: %i",
                    sensor_txt,
                    count_class_0)


        logger.info("Number of %s features with class 1: %i",
                    sensor_txt,
                    count_class_1)

        #### 4.3 Build Samples ###########
        training_images = build_training_img_dict(training_polys, sensor_cfg)

        sample_collection = create_flat_sample( training_images,
                                                training_polys,
                                                training_fall_comp_img
                                                )
        training_bands = sensor_cfg.optical_bands + training_fall_comp_img.bandNames().getInfo()

        classifier = train_rf_classifier(sample_collection,
                                         training_bands,
                                         CFG.rf_params)

        rf_asset_path = save_classifier(classifier,
                                        CFG.gee_asset_root,
                                        count_class_0 + count_class_1,
                                        skey)

        progress(
            "a new %s classifier is being saved as '%s'. Once that task "
            "finishes (Tasks tab at https://code.earthengine.google.com/), "
            "paste that name into section 0 to skip training next time",
            sensor_txt, rf_asset_path.name)

    if classifier is None:
      raise RuntimeError("Classifier was not created or loaded; cannot continue.")

    ###########################################
    # 5 Inference on Target Date Imagery
    ###########################################

    if prior_detections is not None:
        clear_crop_snow_mask = clear_crop_snow_mask.And(no_prior_detections_mask)


    target_image = (get_img_for_date(aoi.date_str,
                                     aoi.geom,
                                     sensor_cfg)
                    .updateMask(clear_crop_snow_mask)
                    .select(sensor_cfg.optical_bands)
                    .addBands(target_fall_comp_img)
                    )

    inference_bands = sensor_cfg.optical_bands + target_fall_comp_img.bandNames().getInfo()

    rf_output = target_image.select(inference_bands).classify(classifier)

    ###########################################
    # 6 Post Processing
    ###########################################

    noise_removed = drop_clusters_below_thresh(rf_output, CFG.noise_thresh)

    cluster_polys = merge_and_clean_clusters(
                                              noise_removed,
                                              CFG.post_kernel,
                                              sensor_cfg.cluster_thresh,
                                              sensor_aoi_coverage.geom)

    ## even though we masked prior detections, due to the dilation/erosion we
    ## need to clean this up a bit more
    if prior_detections is not None:
        cluster_polys_geom = cluster_polys.geometry().dissolve(maxError=1)

        new_detections = cluster_polys_geom.difference(prior_detections_geom, 1)
        new_detections_parts = ee.FeatureCollection(
                                    ee.List(new_detections.geometries())
                                    .map(lambda g: ee.Feature(ee.Geometry(g)))
                                      )

        parts_above_threshold = new_detections_parts.map(
            lambda f: f.set('area_m2', f.geometry().area())
            ).filter(ee.Filter.gte('area_m2', sensor_cfg.min_area_after_subtraction))

        cluster_polys = parts_above_threshold

    # unique run numbers could be here too
    cluster_polys_with_sensor = cluster_polys.map(
        lambda f: f.set('sensor', ee.String(skey))
        )

    sensor_output = SensorOutputs(
        ic = sensor_ic,
        aoi_coverage = sensor_aoi_coverage,
        clearsky = clearsky,
        crop_mask = crop_mask,
        vis_brightness = vis_brightness,
        inference_bands = inference_bands,
        target_image = target_image,
        rf_output = rf_output,
        noise_removed = noise_removed,
        detection_polys = cluster_polys_with_sensor
        )

    sensor_outputs[skey] = sensor_output


detection_fcs: list[ee.FeatureCollection] = [
    output.detection_polys
    for output in sensor_outputs.values()
    if output.detection_polys is not None
]

if len(detection_fcs) == 0:
    logger.info("no images had suitable atmospheric and surface conditions for inferencing")

else:
    # if multiple sensors collect imagery for this date, results are merged
    if len(detection_fcs) > 1:
        logger.info("multiple sensors with usable imagery. merging outputs to single feature collection")

        flat_detections = merge_overlapping_features(
                                          *detection_fcs,
                                          params = CFG.feature_export_params)

    else:
        flat_detections = detection_fcs[0]

    # tag detections that are expansions of previously detected spreads
    if prior_detections is not None:
        flat_detections = flat_detections.map(
            lambda f: f.set(CFG.feature_export_params.prior_spread_adjacency_prop_name,
                            f.geometry().intersects(
                                prior_detections_geom.buffer(10), maxError = 1
                                )))
    else:
        flat_detections = flat_detections.map(
            lambda f: f.set(CFG.feature_export_params.prior_spread_adjacency_prop_name, False))


    flat_detections_with_date = flat_detections.map(
        lambda f: f.set('detection_date', ee.String(aoi.date_str)))


    feature_enricher = make_feature_enricher(params = CFG.feature_export_params,
                                             aoi_context = aoi)

    detections_enriched = flat_detections_with_date.map(feature_enricher)

    detections_with_pretty_ids = set_global_ids(
                                      detections_enriched,
                                      CFG.feature_export_params
    )

    geojson_name = f"{CFG.export_prefix}_{CFG.target_date}"

    task = ee.batch.Export.table.toDrive(
        collection=detections_with_pretty_ids,
        description=geojson_name,
        folder=CFG.table_export_folder,
        fileNamePrefix= geojson_name,
        fileFormat="GeoJSON",
        selectors=[".geo"] + list(CFG.feature_export_params.export_fields)
        )

    task.start()

    logger.info('inferencing and postprocessing tasks initiated in cloud')
    logger.info('check GEE code for progress \n')

    if EXPORT_EXCEL:
        ee.batch.Export.table.toDrive(
            collection=detections_with_pretty_ids,
            description=f"{geojson_name}_table",
            folder=CFG.table_export_folder,
            fileNamePrefix=geojson_name,
            fileFormat="CSV",
            selectors=list(CFG.feature_export_params.export_fields),  # no .geo
        ).start()



## Image Chip Export (Optional)


In [ ]:
if CFG.export_image_chips and geojson_name:

    wait_for_detection_geojson(geojson_name,
                              folder_path=CFG.table_export_path,
                              poll_seconds=CFG.poll_seconds,
                              timeout_minutes=CFG.timeout_minutes,
                              logger=logger
                              )

    detections_gdf = load_newest_geojson_for_date(CFG.target_date,
                                                  CFG.table_export_path,
                                                  CFG.export_prefix)
    if detections_gdf is None:
        logger.warning("No detection GeoJSON available for %s after waiting. Check task status in the GEE Code Editor.",
                      CFG.target_date)

    else:
        logger.info("There are %i features to export", len(detections_gdf))

        s2_detections_gdf = detections_gdf[detections_gdf["sensor"]
                                  .astype("string")
                                  .str.contains("s2", case=False, na=False)]
        s2_count = len(s2_detections_gdf)

        l89_detections = detections_gdf[detections_gdf["sensor"].eq("l89")]
        l89_count = len(l89_detections)

        if s2_count + l89_count != len(detections_gdf):
            logger.warning("sensor labels appear corrupted. check geojson")

        if s2_count > 0:
            logger.info("exporting detections with s2 imagery (%i total)", s2_count)

            s2_detections_fc = geemap.gdf_to_ee(s2_detections_gdf, geodesic=True)
            export_all(s2_detections_fc,
                      image_collection = ics['s2'],
                      export_params = CFG.feature_export_params,
                      sensor_config = SENSORS['s2']
                      )
        else:
            logger.info("no detections with s2 imagery to export")

        if l89_count > 0:
            logger.info("exporting detections with only l89 imagery (%i total)",
                        l89_count)

            l89_detections_fc = geemap.gdf_to_ee(l89_detections, geodesic=True)
            export_all(l89_detections_fc,
                      image_collection = ics['l89'],
                      export_params = CFG.feature_export_params,
                      sensor_config = SENSORS['l89']
                      )
        else:
            logger.info("no detections with only l89 imagery to export")

elif CFG.export_image_chips:
    logger.info("no detections were found, so there are no image chips to export")

else:
    logger.info("image chips not exported. Set EXPORT_IMAGE_CHIPS = True in "
                "section 0 if you want them")
